# NSL-KDD — Exploratory Data Analysis

Goal: understand the feature distribution, class balance, and data-quality issues before training.

This notebook is reproducible — it only reads from `data/raw/` and never modifies it.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is opened from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_train_test, get_dataset_summary

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)

## 1. Load the dataset

In [ ]:
train_df, test_df = load_train_test()
print('Train:', train_df.shape, ' Test:', test_df.shape)
train_df.head()

In [ ]:
get_dataset_summary(train_df, 'Train')

## 2. Binary class balance

In [ ]:
binary_counts = train_df['binary_label'].value_counts(normalize=True)
print(binary_counts)

fig, ax = plt.subplots(figsize=(5, 4))
binary_counts.plot(kind='bar', ax=ax, color=['tab:blue', 'tab:red'])
ax.set_ylabel('Proportion')
ax.set_title('Binary class distribution — NSL-KDD Train')
plt.xticks(rotation=0)
plt.tight_layout();

## 3. Attack-category distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, (train_df, test_df), ('Train', 'Test')):
    df['attack_category'].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(f'Attack categories — {title}')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout();

## 4. Protocol / service / flag cardinality

In [ ]:
for col in ('protocol_type', 'service', 'flag'):
    print(f'{col}: {train_df[col].nunique()} unique values')
train_df['service'].value_counts().head(15)

## 5. Numerical feature summary

In [ ]:
numeric = train_df.select_dtypes(include=[np.number])
numeric.describe().T.head(20)

## 6. Correlation heatmap (numeric features)

Highly correlated features are candidates for dimensionality reduction later. For tree-based models they rarely hurt accuracy, but they inflate training time.

In [ ]:
corr = numeric.corr().abs()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='magma', square=True, cbar_kws={'shrink': .6}, ax=ax)
ax.set_title('Absolute correlation — numeric features')
plt.tight_layout();

## 7. Benign vs Malicious — feature distributions

Short look at a few features whose distribution visibly differs between classes.

In [ ]:
features_to_plot = ['src_bytes', 'dst_bytes', 'count', 'serror_rate']
fig, axes = plt.subplots(1, len(features_to_plot), figsize=(4 * len(features_to_plot), 3.5))
for ax, feat in zip(axes, features_to_plot):
    for label in ('Benign', 'Malicious'):
        vals = np.log1p(train_df.loc[train_df['binary_label'] == label, feat])
        ax.hist(vals, bins=40, alpha=0.55, label=label)
    ax.set_title(feat)
    ax.set_xlabel('log1p(value)')
    ax.legend()
plt.tight_layout();

## 8. Novel attack types in the test set

A known NSL-KDD property: the test set contains attack labels not present in train. This forces the model to generalise.

In [ ]:
train_attacks = set(train_df['label'].str.strip().unique())
test_attacks = set(test_df['label'].str.strip().unique())
novel = sorted(test_attacks - train_attacks)
print(f'Novel attack types in test set: {len(novel)}')
print(novel)